# 2. Replikasi Temuan Bantul pada Sleman

Notebook kedua dari empat notebook **Gabungan**.

Analisis Bantul menghasilkan sejumlah temuan yang menjadi dasar cara seluruh
proyek ini dibaca. Pertanyaan notebook ini sederhana:

> **Temuan mana yang muncul lagi di Sleman, dan mana yang tidak?**

Ini bukan formalitas. Temuan yang muncul lagi pada data kabupaten lain jauh lebih
kuat daripada temuan yang hanya terlihat sekali -- dan temuan yang **tidak**
muncul lagi memberi tahu kita bahwa yang dulu kita kira sifat penyakit ternyata
sifat satu kabupaten.

Setiap bagian di bawah ini menghitung ulang angkanya untuk kedua wilayah,
berdampingan, lalu menyatakan lulus atau tidak. Tidak ada angka yang disalin dari
notebook Bantul; semuanya dihitung ulang di sini dari `dataset_*.csv`.

| # | Temuan yang diuji | Bagian |
|---|---|---|
| 1 | Label Diabetes pada dasarnya adalah aturan `IMT > 25` | 1 |
| 2 | Tinggi badan tidak menaikkan risiko Kanker Paru -- ia hanya menandai jenis kelamin | 2 |
| 3 | Korelasi Usia-IMT berbalik tanda ketika anak dipisahkan | 3 |
| 4 | Kekuatan asosiasi tiap variabel dengan tiap penyakit | 4 |
| 5 | Segmen mana yang sebenarnya mendorong TBC | 5 |
| 6 | Layak usia tidak berarti benar-benar ditanya | 6 |

In [ ]:
# ===================== IMPORTS =====================
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Akar proyek dicari dari folder kerja -- tidak ada jalur absolut di notebook
# ini, jadi seluruh folder boleh dipindah.
AKAR = Path.cwd()
while not (AKAR / "Bersama").exists():
    AKAR = AKAR.parent
sys.path.insert(0, str(AKAR / "Bersama"))

import konfigurasi as cfg
import bmi_standards as bs
import viz_style as vz

vz.apply_style()
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 190)

WILAYAH = ["Bantul", "Sleman"]
WARNA = vz.PALETTE["WILAYAH"]

# Nama pendek -> kolom target, dipakai di seluruh notebook Gabungan.
TARGETS = cfg.DISEASE_TARGETS
NAMA_PENYAKIT = {
    "diabetes": "Diabetes Melitus",
    "hipertensi": "Hipertensi",
    "kanker_paru": "Kanker Paru",
    "tbc": "TBC",
}

# Pita usia tempat tiap kuesioner ditanyakan. WAJIB ikut disebut di setiap
# angka prevalensi: "Tidak Dapat Pertanyaan" adalah gerbang kelayakan, bukan
# data hilang, sehingga tiap penyakit punya subpopulasi yang berbeda.
PITA_USIA = {
    "diabetes": "15+",
    "hipertensi": "15+",
    "kanker_paru": "45+",
    "tbc": "semua usia",
}

print("Wilayah dibandingkan:", ", ".join(WILAYAH))

In [ ]:
# ===================== MEMUAT DATA KEDUA WILAYAH =====================
from scipy import stats

master = {w: cfg.baca_master(w) for w in WILAYAH}
data = {w: {n: cfg.baca_dataset(w, n) for n in TARGETS} for w in WILAYAH}

for w in WILAYAH:
    print(f"{w:<8} master {len(master[w]):>9,} | " +
          " | ".join(f"{n} {len(data[w][n]):,}" for n in TARGETS))


def hasil(nama, bantul, sleman, sama, catatan=""):
    """Satu baris ringkasan replikasi, dikumpulkan lalu ditampilkan di Bagian 7."""
    REPLIKASI.append({
        "Temuan": nama,
        "Bantul": bantul,
        "Sleman": sleman,
        "Replikasi": "YA" if sama else "TIDAK",
        "Catatan": catatan,
    })


REPLIKASI = []

## 1. Apakah Label Diabetes Juga Sekadar Aturan `IMT > 25` di Sleman?

Temuan Bantul: bagi peserta dewasa, `P(Berisiko | IMT > 25)` praktis sama dengan
1. Artinya kuesioner Diabetes tidak mengukur penyakit -- ia menjalankan aturan
IMT. Model yang "berhasil" memprediksi Diabetes sebenarnya hanya menemukan
kembali aturan tersebut, dan akurasinya **tidak boleh** disajikan sebagai
keberhasilan prediksi.

Batas usia yang dipakai adalah `Usia > 20`, mengikuti `bs.CHILD_MAX_AGE`: di
bawah itu klasifikasi memakai persentil CDC, yang aturannya memang berbeda.

In [ ]:
# ===================== ATURAN IMT > 25 PADA DIABETES =====================
aturan = []
for w in WILAYAH:
    d = data[w]["diabetes"]
    kol = TARGETS["diabetes"]

    dewasa = d[d["Usia"] > bs.CHILD_MAX_AGE]
    atas = dewasa[dewasa["IMT_recalculated"] > 25]
    bawah = dewasa[dewasa["IMT_recalculated"] <= 25]

    remaja = d[(d["Usia"] >= 15) & (d["Usia"] <= bs.CHILD_MAX_AGE)]
    remaja_atas = remaja[remaja["IMT_recalculated"] > 25]

    aturan.append({
        "Wilayah": w,
        "n dewasa IMT>25": f"{len(atas):,}",
        "P(Berisiko | IMT>25)": round(atas[kol].mean(), 4),
        "Pengecualian": f"{int((atas[kol] == 0).sum()):,}",
        "P remaja 15-20": round(remaja_atas[kol].mean(), 4),
        "P(Berisiko | IMT<=25)": round(bawah[kol].mean(), 4),
    })

tabel = pd.DataFrame(aturan).set_index("Wilayah")
display(tabel)

for r in aturan:
    print(f"{r['Wilayah']:<8} {r['Pengecualian']:>6} pengecualian dari "
          f"{r['n dewasa IMT>25']:>9} baris ber-IMT di atas 25")

hasil("Diabetes = aturan IMT > 25",
      f"P = {tabel.loc['Bantul', 'P(Berisiko | IMT>25)']:.4f}",
      f"P = {tabel.loc['Sleman', 'P(Berisiko | IMT>25)']:.4f}",
      bool((tabel["P(Berisiko | IMT>25)"] > 0.99).all()),
      "aturan kuesioner, bukan prediksi penyakit")

In [ ]:
# ===================== SISA SINYAL DI BAWAH IMT 25 =====================
# Di ATAS 25 labelnya sudah ditentukan aturan. Sinyal yang sungguh-sungguh ada
# justru di BAWAH 25, dan di sana prevalensi naik menurut usia di kedua wilayah.
BINS = [15, 30, 40, 50, 60, 70, 120]
LBL = ["15-29", "30-39", "40-49", "50-59", "60-69", "70+"]

bawah_usia = {}
for w in WILAYAH:
    d = data[w]["diabetes"]
    b = d[(d["Usia"] > bs.CHILD_MAX_AGE) & (d["IMT_recalculated"] <= 25)].copy()
    b["pita"] = pd.cut(b["Usia"], bins=BINS, labels=LBL, right=False)
    bawah_usia[w] = (b.groupby("pita", observed=True)[TARGETS["diabetes"]]
                       .mean().mul(100).round(2))

banding = pd.DataFrame(bawah_usia)
display(banding)

fig, ax = plt.subplots(figsize=vz.FIG_SINGLE)
for w in WILAYAH:
    ax.plot(banding.index.astype(str), banding[w], marker="o", color=WARNA[w],
            label=w)
ax.set_ylabel("% Berisiko Diabetes")
ax.set_xlabel("Kelompok usia")
ax.set_title("Sisa sinyal nyata: peserta dewasa ber-IMT <= 25\n"
             "(di atas IMT 25 label sudah ditentukan aturan)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Replikasi: ya.** Nilai P di kedua wilayah berada di atas 0,99, dengan jumlah
pengecualian yang sangat kecil dibanding jumlah barisnya. Kesimpulan Bantul
berlaku penuh di Sleman: **model Diabetes menyusun ulang aturan penilaian
kuesioner, bukan memprediksi penyakit.**

Sleman punya pengecualian lebih banyak daripada Bantul dalam angka absolut,
tetapi proporsinya tetap di bawah 0,2%. Itu tidak mengubah kesimpulan apa pun --
aturannya tetap aturan.

Grafik kedua adalah bagian yang benar-benar informatif: di bawah IMT 25, tempat
aturan tidak lagi menentukan jawaban, prevalensi naik menurut usia di **kedua**
wilayah dan dengan bentuk yang sama.

## 2. Tinggi Badan dan Kanker Paru: Perancu Terbersih di Proyek Ini

Temuan Bantul: secara gabungan, tinggi badan tampak berhubungan positif dengan
risiko Kanker Paru. Begitu laki-laki dan perempuan **dipisah**, hubungan itu
hilang sama sekali.

Penjelasannya: laki-laki lebih tinggi, dan laki-laki yang membawa risikonya
(kuesioner Kanker Paru sangat dipengaruhi status merokok, yang tidak ada di data
ini). Tinggi badan hanya **menandai jenis kelamin**.

> **Jangan pernah menulis bahwa orang yang lebih tinggi berisiko kanker paru
> lebih besar.** Ini pemeriksaan yang membuktikan kenapa.

In [ ]:
# ===================== TINGGI -> KANKER PARU, DIPISAH JENIS KELAMIN =====================
tinggi_kp = []
for w in WILAYAH:
    d = data[w]["kanker_paru"]
    kol = TARGETS["kanker_paru"]
    baris = {"Wilayah": w,
             "r gabungan": round(float(stats.pointbiserialr(d[kol], d["Tinggibadan"]).statistic), 3)}
    for jk, label in [("L", "laki-laki"), ("P", "perempuan")]:
        sub = d[d["Jeniskelamin"] == jk]
        baris[f"r {label}"] = round(
            float(stats.pointbiserialr(sub[kol], sub["Tinggibadan"]).statistic), 3)
        baris[f"% berisiko {label}"] = round(sub[kol].mean() * 100, 2)
        baris[f"tinggi rata2 {label}"] = round(sub["Tinggibadan"].mean(), 1)
    tinggi_kp.append(baris)

tabel_kp = pd.DataFrame(tinggi_kp).set_index("Wilayah")
display(tabel_kp[["r gabungan", "r laki-laki", "r perempuan"]])
display(tabel_kp[["% berisiko laki-laki", "% berisiko perempuan",
                  "tinggi rata2 laki-laki", "tinggi rata2 perempuan"]])

hasil("Tinggi -> Kanker Paru runtuh setelah jenis kelamin dipisah",
      f"{tabel_kp.loc['Bantul', 'r gabungan']:+.3f} -> "
      f"{tabel_kp.loc['Bantul', 'r laki-laki']:+.3f} / {tabel_kp.loc['Bantul', 'r perempuan']:+.3f}",
      f"{tabel_kp.loc['Sleman', 'r gabungan']:+.3f} -> "
      f"{tabel_kp.loc['Sleman', 'r laki-laki']:+.3f} / {tabel_kp.loc['Sleman', 'r perempuan']:+.3f}",
      bool((tabel_kp[["r laki-laki", "r perempuan"]].abs() < 0.10).all().all()),
      "tinggi badan hanya menandai jenis kelamin")

In [ ]:
# ===================== BUKTI VISUAL: DESIL TINGGI BADAN =====================
fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2, sharey=True)

for ax, w in zip(axes, WILAYAH):
    d = data[w]["kanker_paru"].copy()
    kol = TARGETS["kanker_paru"]
    d["desil"] = pd.qcut(d["Tinggibadan"], 10, duplicates="drop")

    # Garis gabungan: naik tajam -- inilah kesan yang menyesatkan.
    g = d.groupby("desil", observed=True).agg(
        x=("Tinggibadan", "mean"), y=(kol, "mean"))
    ax.plot(g["x"], g["y"] * 100, marker="o", color="black", lw=2.5,
            label="Gabungan (menyesatkan)")

    # Dipisah jenis kelamin: dua garis datar.
    for jk, label in [("L", "Laki-laki"), ("P", "Perempuan")]:
        s = d[d["Jeniskelamin"] == jk]
        gs = s.groupby("desil", observed=True).agg(
            x=("Tinggibadan", "mean"), y=(kol, "mean")).dropna()
        ax.plot(gs["x"], gs["y"] * 100, marker="s", ls="--",
                color=vz.PALETTE["SEX"][jk], label=label)

    ax.set_title(f"{w}  (usia 45+, n = {len(d):,})")
    ax.set_xlabel("Tinggi badan rata-rata desil (cm)")
    ax.grid(alpha=0.3)
    ax.legend()

axes[0].set_ylabel("% Berisiko Kanker Paru")
fig.suptitle("Garis hitam naik hanya karena desil tinggi berisi makin banyak "
             "laki-laki", fontsize=12)
plt.tight_layout()
plt.show()

**Replikasi: ya, dan sama tegasnya.** Korelasi gabungan yang cukup besar runtuh
menjadi mendekati nol di dalam masing-masing jenis kelamin, di **kedua** wilayah.
Selisih prevalensi antara laki-laki dan perempuan tetap sangat besar di keduanya.

Grafik desil memperlihatkan mekanismenya secara langsung: garis hitam naik tajam,
tetapi dua garis putus-putus di dalamnya nyaris mendatar. Yang naik sepanjang
sumbu-X bukan risiko, melainkan **proporsi laki-laki di dalam desil itu**.

## 3. Korelasi Usia-IMT Berbalik Tanda

Temuan Bantul: digabung seluruh usia, Usia dan IMT berkorelasi positif cukup
kuat. Dibatasi pada dewasa saja, korelasinya menjadi **negatif**.

Penyebabnya bukan penuaan melainkan **pertumbuhan anak**: dari usia 0 sampai 20
tahun, IMT memang naik seiring usia karena tubuhnya sedang tumbuh. Korelasi
gabungan sebagian besar mengukur hal itu.

Ini contoh paling jelas bahwa satu angka korelasi gabungan dapat menyembunyikan
dua pola yang berlawanan.

In [ ]:
# ===================== USIA vs IMT: GABUNGAN vs DEWASA =====================
usia_imt = []
for w in WILAYAH:
    d = master[w]
    dewasa = d[d["Usia"] > bs.CHILD_MAX_AGE]
    anak = d[d["Usia"] <= bs.CHILD_MAX_AGE]
    usia_imt.append({
        "Wilayah": w,
        "r semua usia": round(d["Usia"].corr(d["IMT_recalculated"]), 3),
        "r anak (0-20)": round(anak["Usia"].corr(anak["IMT_recalculated"]), 3),
        "r dewasa (>20)": round(dewasa["Usia"].corr(dewasa["IMT_recalculated"]), 3),
        "n dewasa": f"{len(dewasa):,}",
    })

tabel_ui = pd.DataFrame(usia_imt).set_index("Wilayah")
display(tabel_ui)

fig, ax = plt.subplots(figsize=vz.FIG_SINGLE)
for w in WILAYAH:
    d = master[w]
    g = d.groupby("Usia")["IMT_recalculated"].agg(["mean", "size"])
    g = g[g["size"] >= 100]
    ax.plot(g.index, g["mean"], color=WARNA[w], label=w, lw=2)
ax.axvline(bs.CHILD_MAX_AGE, color="black", ls="--", lw=1)
ax.text(bs.CHILD_MAX_AGE + 0.8, ax.get_ylim()[1] * 0.97,
        f"usia {bs.CHILD_MAX_AGE}: batas anak/dewasa", fontsize=9, va="top")
ax.set_xlabel("Usia (tahun)")
ax.set_ylabel("IMT rata-rata")
ax.set_title("IMT rata-rata menurut usia -- naik tajam pada anak, "
             "landai lalu turun pada dewasa")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

hasil("Korelasi Usia-IMT berbalik tanda pada dewasa",
      f"{tabel_ui.loc['Bantul', 'r semua usia']:+.3f} -> {tabel_ui.loc['Bantul', 'r dewasa (>20)']:+.3f}",
      f"{tabel_ui.loc['Sleman', 'r semua usia']:+.3f} -> {tabel_ui.loc['Sleman', 'r dewasa (>20)']:+.3f}",
      bool((tabel_ui["r semua usia"] > 0).all() and (tabel_ui["r dewasa (>20)"] < 0).all()),
      "korelasi gabungan mengukur pertumbuhan anak")

## 4. Tabel Kekuatan Asosiasi

Ukuran yang dipakai persis sama dengan yang dipakai notebook pemodelan tiap
wilayah, supaya angkanya bisa dibandingkan langsung:

- **Variabel numerik** -> korelasi *point-biserial* (bertanda, sehingga arah
  hubungan ikut terbaca)
- **Variabel kategorik** -> *Cramer's V* (selalu positif: hanya kekuatan, tanpa
  arah)

Semua dihitung **hanya pada baris yang benar-benar ditanyai** untuk target
tersebut.

In [ ]:
# ===================== TABEL ASOSIASI KEDUA WILAYAH =====================
NUM = ["Usia", "Tinggibadan", "Berat Badan", "IMT_recalculated"]
CAT = ["Jeniskelamin", "segmen"]


def cramers_v(x, y):
    tab = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(tab)[0]
    n = tab.to_numpy().sum()
    r, k = tab.shape
    return float(np.sqrt((chi2 / n) / min(r - 1, k - 1)))


asosiasi = {}
for w in WILAYAH:
    baris = {}
    for nama, kol in TARGETS.items():
        d = data[w][nama]
        nilai = {c: float(stats.pointbiserialr(d[kol], d[c]).statistic) for c in NUM}
        nilai.update({c: cramers_v(d[c], d[kol]) for c in CAT})
        baris[NAMA_PENYAKIT[nama]] = nilai
    asosiasi[w] = pd.DataFrame(baris).T[NUM + CAT]

for w in WILAYAH:
    print(f"--- {w} ---")
    display(asosiasi[w].round(3))

print("Variabel terkuat per penyakit (nilai mutlak):")
terkuat = {}
for w in WILAYAH:
    terkuat[w] = asosiasi[w].abs().idxmax(axis=1)
display(pd.DataFrame(terkuat))

In [ ]:
# ===================== SELISIH ANTAR WILAYAH =====================
selisih = (asosiasi["Sleman"] - asosiasi["Bantul"]).round(3)
display(selisih.style.background_gradient(cmap="RdBu_r", vmin=-0.1, vmax=0.1)
        if hasattr(selisih, "style") else selisih)

sama = (terkuat["Bantul"] == terkuat["Sleman"])
print("\nVariabel terkuat sama di kedua wilayah:")
for p in sama.index:
    tanda = "SAMA " if sama[p] else "BEDA "
    print(f"   {tanda} {p:<18} Bantul: {terkuat['Bantul'][p]:<18} "
          f"Sleman: {terkuat['Sleman'][p]}")

print(f"\nSelisih terbesar: {selisih.abs().to_numpy().max():.3f}")

hasil("Peringkat variabel terkuat per penyakit",
      ", ".join(f"{p.split()[0]}={v}" for p, v in terkuat["Bantul"].items()),
      ", ".join(f"{p.split()[0]}={v}" for p, v in terkuat["Sleman"].items()),
      bool(sama.all()),
      "empat penyakit, variabel terkuat yang sama")

Dua hal yang perlu dibaca dari tabel ini, dan keduanya berlaku di kedua wilayah:

**Angka terbesar di tabel adalah yang paling tidak informatif.** Asosiasi
IMT-Diabetes adalah yang tertinggi, tetapi ia melingkar: labelnya memang dibuat
dari IMT (Bagian 1). Angka yang jauh lebih layak dilaporkan adalah
IMT-Hipertensi, karena kurva prevalensinya naik **mulus**, bukan seperti tangga
-- bentuk yang memang diharapkan dari hubungan biologis yang nyata.

**TBC tidak dijelaskan dengan baik oleh variabel mana pun.** Nilai tertingginya
jauh di bawah penyakit lain di kedua wilayah. Itu konsisten dengan lemahnya
performa model TBC nanti: keterbatasannya ada pada **variabel yang tersedia**,
bukan pada pilihan algoritma.

## 5. Segmen Mana yang Sebenarnya Mendorong TBC?

Temuan Bantul yang paling berlapis, dan yang paling perlu diperiksa ulang.

Pada Bantul, `BP PN` punya prevalensi TBC tertinggi -- tetapi itu **efek usia**:
sebagian besar anggotanya pensiunan berusia 60 tahun ke atas. Setelah dipisah per
pita usia, `PBPU` yang unggul di hampir setiap pita, pada usia rata-rata jauh
lebih muda. Jadi ada tiga jawaban berbeda tergantung pertanyaannya:

- prevalensi tertinggi apa adanya -> `BP PN`
- prevalensi tertinggi yang bukan sekadar usia -> `PBPU`
- penyumbang catatan berisiko terbanyak -> `APBN`, semata-mata karena ukurannya

Bagian ini juga tempat satu aturan lama harus ditulis ulang. Pada Bantul,
`BP SWASTA` hanya punya 409 baris dan **tidak boleh dilaporkan**. Di Sleman
jumlahnya jauh lebih besar.

In [ ]:
# ===================== TBC PER SEGMEN =====================
per_segmen = {}
for w in WILAYAH:
    d = data[w]["tbc"]
    kol = TARGETS["tbc"]
    dasar = d[kol].mean()
    g = d.groupby("segmen").agg(
        n=(kol, "size"), berisiko=(kol, "sum"), usia_rata=("Usia", "mean"))
    g["% Berisiko"] = (g["berisiko"] / g["n"] * 100).round(2)
    g["RR"] = (g["berisiko"] / g["n"] / dasar).round(2)
    g["usia_rata"] = g["usia_rata"].round(1)
    g["% dari semua penanda"] = (g["berisiko"] / g["berisiko"].sum() * 100).round(1)
    per_segmen[w] = g.sort_values("% Berisiko", ascending=False)

    print(f"--- {w} --- (dasar populasi {dasar * 100:.2f}%, "
          f"{int(g['berisiko'].sum()):,} catatan berisiko)")
    display(per_segmen[w][["n", "% Berisiko", "RR", "usia_rata",
                           "% dari semua penanda"]])

In [ ]:
# ===================== SETELAH USIA DIKENDALIKAN =====================
# Inilah pemeriksaan yang membedakan "segmen berisiko tinggi" dari
# "segmen berisi orang tua". Sebuah segmen baru layak disebut punya pengaruh
# nyata bila ia tetap tinggi DI DALAM pita usia.
BINS_S = [0, 15, 30, 45, 60, 120]
LBL_S = ["0-14", "15-29", "30-44", "45-59", "60+"]

per_pita = {}
for w in WILAYAH:
    d = data[w]["tbc"].copy()
    kol = TARGETS["tbc"]
    d["pita"] = pd.cut(d["Usia"], bins=BINS_S, labels=LBL_S, right=False)
    t = d.pivot_table(index="segmen", columns="pita", values=kol,
                      aggfunc="mean", observed=True).mul(100).round(2)
    n = d.pivot_table(index="segmen", columns="pita", values=kol,
                      aggfunc="size", observed=True)
    # Sel dengan sedikit sekali baris tidak dilaporkan: 40% dari 5 orang bukan
    # 40% risiko. Inilah aturan yang dulu ditulis khusus untuk BP SWASTA.
    t = t.mask(n < 100)
    per_pita[w] = t.reindex(per_segmen[w].index)
    print(f"--- {w} --- prevalensi TBC (%) per segmen x pita usia "
          f"(sel dengan n < 100 dikosongkan)")
    display(per_pita[w])

In [ ]:
# ===================== SEGMEN MANA YANG UNGGUL DI DALAM PITA USIA =====================
ringkas_seg = []
for w in WILAYAH:
    t = per_pita[w]
    menang = t.idxmax()                      # segmen teratas tiap pita usia
    ringkas_seg.append({
        "Wilayah": w,
        "Tertinggi apa adanya": per_segmen[w].index[0],
        "Teratas terbanyak dalam pita usia": menang.value_counts().idxmax(),
        "dari berapa pita": f"{menang.value_counts().max()}/{menang.notna().sum()}",
        "Penyumbang penanda terbanyak":
            per_segmen[w]["% dari semua penanda"].idxmax(),
    })
    print(f"{w}: segmen teratas tiap pita usia -> {menang.to_dict()}")

display(pd.DataFrame(ringkas_seg).set_index("Wilayah"))

fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2, sharey=True)
for ax, w in zip(axes, WILAYAH):
    t = per_pita[w]
    for seg in t.index:
        ax.plot(t.columns.astype(str), t.loc[seg], marker="o", label=seg)
    ax.set_title(f"{w}")
    ax.set_xlabel("Pita usia")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("% Berisiko TBC")
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
fig.suptitle("Prevalensi TBC per segmen, DI DALAM pita usia", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ===================== UKURAN SAMPEL BP SWASTA =====================
# Aturan lama "jangan laporkan BP SWASTA" ditulis untuk Bantul, tempat segmen
# ini hanya berisi beberapa ratus baris. Aturannya harus dinyatakan sebagai
# syarat ukuran sampel per wilayah, bukan sebagai larangan tetap.
cek_kecil = []
for w in WILAYAH:
    g = per_segmen[w]
    for seg in g.index:
        n = int(g.loc[seg, "n"])
        cek_kecil.append({
            "Wilayah": w, "segmen": seg, "n": f"{n:,}",
            "% Berisiko": g.loc[seg, "% Berisiko"],
            "Layak dilaporkan?": "ya" if n >= 1000 else "TIDAK (n terlalu kecil)",
        })
kecil = pd.DataFrame(cek_kecil)
display(kecil[kecil["Layak dilaporkan?"] != "ya"])
print("Segmen yang tidak layak dilaporkan per wilayah:")
for w in WILAYAH:
    sub = kecil[(kecil["Wilayah"] == w) & (kecil["Layak dilaporkan?"] != "ya")]
    print(f"   {w:<8} {list(sub['segmen']) or 'tidak ada'}")

hasil("PBPU adalah sinyal segmen yang nyata pada TBC",
      ringkas_seg[0]["Teratas terbanyak dalam pita usia"],
      ringkas_seg[1]["Teratas terbanyak dalam pita usia"],
      ringkas_seg[0]["Teratas terbanyak dalam pita usia"]
      == ringkas_seg[1]["Teratas terbanyak dalam pita usia"],
      "dibandingkan DI DALAM pita usia, bukan apa adanya")

**Replikasi: ya, tetapi lebih berantakan di Sleman -- dan itu perlu dikatakan.**

Yang bertahan utuh adalah **mekanismenya**. Di kedua wilayah, segmen dengan
prevalensi tertinggi apa adanya bukanlah segmen dengan pengaruh terbesar:
peringkatnya berubah begitu usia dikendalikan. Dan di kedua wilayah `PBPU`
adalah segmen yang paling sering unggul di dalam pita usia, pada usia rata-rata
yang jauh lebih muda daripada pesaingnya -- bacaan sosial-ekonomi, bukan efek
usia. `APBN` juga tetap penyumbang catatan berisiko terbanyak di kedua wilayah,
semata-mata karena ukurannya, meski laju risikonya justru di bawah rata-rata.

Yang berubah adalah **siapa yang menempati puncak tabel mentah**:

| | Bantul | Sleman |
|---|---|---|
| Tertinggi apa adanya | `BP PN` | **`BP SWASTA`** |
| Teratas terbanyak dalam pita usia | `PBPU` (4 dari 5 pita) | `PBPU` (3 dari 5 pita) |
| Penyumbang penanda terbanyak | `APBN` | `APBN` |

Keunggulan `PBPU` **melemah** di Sleman: ia kalah pada pita `45-59` (oleh
`BP PN`) dan pada pita `60+` (oleh `BP SWASTA`). Jadi kesimpulannya bertahan,
tetapi tidak sekuat di Bantul, dan tidak boleh dikutip seolah-olah `PBPU`
unggul di setiap pita usia.

**`BP SWASTA` inilah kasus yang memaksa aturan lama ditulis ulang.** Di Bantul
segmen ini hanya berisi 409 baris dan memang tidak layak dilaporkan. Di Sleman
ia berisi 1.695 baris, cukup untuk dilaporkan secara keseluruhan -- dan langsung
menempati puncak tabel. Namun perhatikan bahwa tiga pita usia termudanya tetap
kosong pada tabel silang di atas karena masing-masing berisi kurang dari 100
baris. Jadi segmen ini layak dilaporkan **sebagai satu angka**, tetapi belum
layak dipecah per pita usia.

## 6. Layak Usia Tidak Berarti Benar-benar Ditanya

Temuan Bantul: gerbang usia menentukan **batas** siapa yang boleh ditanya, tetapi
tidak semua orang yang layak usia benar-benar menerima seluruh pertanyaan. Dan
yang tidak menerimanya **bukan kelompok acak** -- mereka jauh lebih sering
ditandai berisiko TBC.

Setiap catatan diberi pola empat bit: `1` bila pertanyaannya benar-benar dijawab,
`0` bila jawabannya `Tidak Dapat Pertanyaan`. Urutannya
**Diabetes - Hipertensi - Kanker Paru - TBC**.

In [ ]:
# ===================== SENSUS POLA PERTANYAAN =====================
URUT = ["diabetes", "hipertensi", "kanker_paru", "tbc"]

pola_wilayah = {}
for w in WILAYAH:
    d = master[w].copy()
    bit = [(d[TARGETS[n]] != "Tidak Dapat Pertanyaan").astype(int).astype(str)
           for n in URUT]
    d["pola"] = bit[0] + bit[1] + bit[2] + bit[3]
    d["berisiko_tbc"] = (d[TARGETS["tbc"]] == "Berisiko").astype(int)

    g = d.groupby("pola").agg(
        n=("pola", "size"), usia_min=("Usia", "min"), usia_maks=("Usia", "max"),
        usia_rata=("Usia", "mean"), tbc=("berisiko_tbc", "mean"))
    g["usia_rata"] = g["usia_rata"].round(1)
    g["% TBC"] = (g["tbc"] * 100).round(2)
    g["% dari total"] = (g["n"] / len(d) * 100).round(2)
    pola_wilayah[w] = g.sort_values("n", ascending=False)

    print(f"--- {w} --- total {len(d):,}")
    display(pola_wilayah[w][["n", "% dari total", "usia_min", "usia_maks",
                             "usia_rata", "% TBC"]])

In [ ]:
# ===================== LENGKAP vs TIDAK LENGKAP PADA USIA 45+ =====================
# Usia 45+ adalah satu-satunya kelompok yang LAYAK menerima keempat pertanyaan,
# jadi hanya di sana "tidak lengkap" punya arti.
lengkap = []
for w in WILAYAH:
    d = master[w].copy()
    bit = [(d[TARGETS[n]] != "Tidak Dapat Pertanyaan").astype(int).astype(str)
           for n in URUT]
    d["pola"] = bit[0] + bit[1] + bit[2] + bit[3]
    d["berisiko_tbc"] = (d[TARGETS["tbc"]] == "Berisiko").astype(int)

    tua = d[d["Usia"] >= 45]
    penuh = tua[tua["pola"] == "1111"]
    kurang = tua[tua["pola"] != "1111"]
    lengkap.append({
        "Wilayah": w,
        "n usia 45+": f"{len(tua):,}",
        "% menerima 4 pertanyaan": round(len(penuh) / len(tua) * 100, 2),
        "% TBC bila lengkap": round(penuh["berisiko_tbc"].mean() * 100, 2),
        "% TBC bila tidak lengkap": round(kurang["berisiko_tbc"].mean() * 100, 2),
        "Rasio": round(kurang["berisiko_tbc"].mean() / penuh["berisiko_tbc"].mean(), 2),
        "usia rata2 lengkap": round(penuh["Usia"].mean(), 1),
        "usia rata2 tidak lengkap": round(kurang["Usia"].mean(), 1),
    })

tabel_l = pd.DataFrame(lengkap).set_index("Wilayah")
display(tabel_l)

hasil("Data yang tidak lengkap bersifat informatif, bukan acak",
      f"{tabel_l.loc['Bantul', 'Rasio']:.2f}x lebih sering ditandai TBC",
      f"{tabel_l.loc['Sleman', 'Rasio']:.2f}x lebih sering ditandai TBC",
      bool((tabel_l["Rasio"] > 1.5).all()),
      "analisis lintas target akan MENGECILKAN jumlah penanda ganda")

**Konsekuensi yang harus ikut dilaporkan.** Karena catatan yang kehilangan blok
Diabetes/Hipertensi jauh lebih sering ditandai TBC, setiap analisis yang hanya
memakai catatan lengkap akan **secara sistematis mengecilkan** jumlah orang yang
ditandai lebih dari satu penyakit.

Arah biasnya diketahui, dan itulah yang membuatnya layak dilaporkan alih-alih
membatalkan analisisnya. Tetapi arah itu harus disebut setiap kali angka lintas
target ditampilkan.

## 7. Ringkasan Replikasi

In [ ]:
# ===================== RINGKASAN =====================
ringkasan = pd.DataFrame(REPLIKASI)
display(ringkasan)

n_ya = int((ringkasan["Replikasi"] == "YA").sum())
print(f"{n_ya} dari {len(ringkasan)} temuan Bantul muncul kembali di Sleman.")
if n_ya < len(ringkasan):
    print("\nTidak replikasi:")
    display(ringkasan[ringkasan["Replikasi"] != "YA"])

### Yang berubah, dan karenanya harus ditulis ulang

**Aturan `BP SWASTA` bukan aturan tetap, melainkan syarat ukuran sampel.**
Catatan lama berbunyi "jangan laporkan `BP SWASTA`" karena di Bantul segmen itu
hanya berisi beberapa ratus baris, sehingga persentasenya dihitung dari segelintir
orang. Di Sleman jumlahnya jauh lebih besar. Bentuk aturan yang benar:

> Jangan melaporkan prevalensi sebuah segmen bila jumlah barisnya kecil --
> **periksa n-nya di wilayah yang sedang dibahas**, jangan mewarisi daftar
> segmen terlarang dari wilayah lain.

Aturan yang sama dipakai pada tabel silang segmen x pita usia di Bagian 5, tempat
sel dengan n < 100 sengaja dikosongkan alih-alih menampilkan persentase dari
beberapa orang saja.

### Yang tidak berubah

Empat temuan inti muncul kembali utuh: Diabetes tetap aturan `IMT > 25`, tinggi
badan tetap hanya menandai jenis kelamin pada Kanker Paru, korelasi Usia-IMT
tetap berbalik tanda, dan ketidaklengkapan kuesioner tetap bersifat informatif.

Semua itu awalnya dirumuskan dari satu kabupaten. Kini masing-masing sudah
terlihat di dua kabupaten, dengan data yang dikumpulkan terpisah -- yang membuat
keempatnya jauh lebih layak dipertahankan di dalam laporan.